# Cohort models 01: Build a multi-image context

This tutorial combines `example_1` and `example_2` without merging their files. Each image retains its own reader, binner, annotation reader, and spectrum numbering. `CohortPixelDataset` presents the members as one dataset while preserving the originating image key for every sample.

## Locate the tutorial workspace

Both example datasets must be present under `data/tutorial_workspace/datasets`. The tutorial-data helper reports a clear error when either directory is missing.

In [ ]:
import os
from pathlib import Path

repository_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)
os.chdir(repository_root)
workspace_root = repository_root / "data" / "tutorial_workspace"
image_paths = {
    name: workspace_root / "datasets" / name / f"{name}.imzML"
    for name in ("example_1", "example_2")
}
assert all(path.is_file() for path in image_paths.values()), (
    "Run docs/tutorials/download_tutorial_data.py first."
)
image_paths

## Configure each image context

A cohort references already configured local contexts. Configure components against each explicit imzML path; this creates independent ledger entries named `example_1` and `example_2`. Annotation discovery remains image-local, so a CSV or SQLite annotation source attached to one member does not leak into the other.

In [ ]:
from msi_autoencoder_wrapper.core.wrapper import MSIAutoEncoderWrapper

wrapper = MSIAutoEncoderWrapper(project_path=str(workspace_root))
for image_path in image_paths.values():
    wrapper.context_manager.set_reader("PyImzMLReader", str(image_path))
    wrapper.context_manager.set_binner(
        "LinearBinning",
        str(image_path),
        bin_step=0.1,
    )

wrapper.context_manager.get_context_config("example_1")

## Create and activate the cohort

Creating a cohort does not replace the active single-image context. Activation changes the execution scope to `cohort`; `workspace.use_image(...)` can still temporarily enter one member and restores the cohort scope afterwards.

In [ ]:
wrapper.cohorts.create("examples")
cohort = wrapper.cohorts.set_images(
    ["example_1", "example_2"],
    name="examples",
)
wrapper.cohorts.activate("examples")
print(wrapper.workspace.execution_scope)
print([member.image_key for member in cohort.members])

## Inspect the cohort dataset

The first field returned for a sample is `(image_key, local_spectrum_id)`. This identity must be retained by analyses and split manifests. Use a grouped split by `image_key` when validation images must be disjoint from training images; a random pixel split answers a different question.

In [ ]:
from msi_autoencoder_wrapper.models.datasets import CohortPixelDataset

dataset = CohortPixelDataset(
    cohort,
    normalization="tic",
    split={
        "strategy": "grouped",
        "seed": 1912,
        "parameters": {"group_fields": "image_key"},
    },
)
print("samples:", len(dataset))
print("first sample:", dataset.get_sample_id(0))
print("last sample:", dataset.get_sample_id(len(dataset) - 1))

## Persist the cohort definition

Saving writes `models/cohort_examples/cohort.json`. The file stores member context snapshots and model-reference policy, not copies of MSI data. A restored cohort resolves those snapshots against the workspace.

In [ ]:
cohort_path = wrapper.cohorts.save("examples")
print(cohort_path)
print(cohort.get_config())

# CohortPixelDataset is selected through the model manager for training:
# wrapper.models_manager.set_dataset(
#     "CohortPixelDataset",
#     normalization="tic",
#     split={
#         "strategy": "grouped",
#         "seed": 1912,
#         "parameters": {"group_fields": "image_key"},
#     },
# )

## Continue with model training

Architecture configuration, compilation, resource estimation, and training use the same stages as [Autoencoder 01](../autoencoder/autoencoder_01_model_configuration_and_training.ipynb). The important difference is dataset selection and split policy; a cohort is an execution context, not a separate neural-network architecture family.